
# Taylor Swift Reddit Analysis

This Google Colab notebook continues the technical work from the cleaned Reddit RSS dataset. It produces:

1. final dataset summary statistics;
2. five exploratory analysis visuals;
3. VADER sentiment analysis;
4. LDA topic modelling using 3, 5 and 10 topic options;
5. five text-mining outputs/visuals for the report.

**Case study:** Taylor Swift  
**Platform:** Reddit  
**Dataset:** cleaned Reddit comments collected through public Reddit RSS feeds.


In [ ]:

# ============================================================
# Cell 1: Install/import libraries and create output folders
# ============================================================

import os
import re
import warnings
from collections import Counter
from textwrap import shorten

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation

import nltk

# NLTK downloads needed in Google Colab.
# punkt_tab and averaged_perceptron_tagger_eng are included because newer NLTK versions may require them.
for package in [
    "vader_lexicon",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "punkt",
    "punkt_tab",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
]:
    try:
        nltk.download(package, quiet=True)
    except Exception as e:
        print(f"Could not download {package}: {e}")

from nltk.sentiment.vader import SentimentIntensityAnalyzer

os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

print("Libraries loaded and folders created: figures/ and outputs/")


In [ ]:

# ============================================================
# Cell 2: Load the cleaned dataset and summary file
# ============================================================

DATA_FILE = "taylor_swift_reddit_cleaned_dataset.csv"
SUMMARY_FILE = "taylor_swift_reddit_dataset_summary.csv"


def find_file(filename):
    """Find a file in common Colab/local locations. If not found in Colab, ask the user to upload it."""
    candidate_paths = [
        filename,
        f"/content/{filename}",
        f"/mnt/data/{filename}",
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            return path

    # Colab upload fallback
    try:
        from google.colab import files
        print(f"Please upload: {filename}")
        uploaded = files.upload()
        if filename in uploaded:
            return filename
        # Fallback to first uploaded CSV if exact name is different
        csv_files = [name for name in uploaded.keys() if name.endswith(".csv")]
        if csv_files:
            return csv_files[0]
    except Exception:
        pass

    raise FileNotFoundError(f"Could not find {filename}. Please upload it to Colab.")


data_path = find_file(DATA_FILE)
df = pd.read_csv(data_path)

try:
    summary_path = find_file(SUMMARY_FILE)
    summary_df = pd.read_csv(summary_path)
except Exception:
    summary_df = pd.DataFrame(columns=["Metric", "Value"])

print(f"Loaded cleaned dataset from: {data_path}")
print("Dataset shape:", df.shape)
print("Columns:")
print(list(df.columns))

if not summary_df.empty:
    print("\nProvided dataset summary:")
    display(summary_df)


In [ ]:

# ============================================================
# Cell 3: Prepare analysis fields and inspect data quality
# ============================================================

# Ensure key text columns exist and contain strings
for col in ["comment_body", "text_readable", "cleaned_text", "post_title", "comment_author_id", "post_link"]:
    if col not in df.columns:
        df[col] = ""
    df[col] = df[col].fillna("").astype(str)

# Parse available dates. Many Reddit RSS comment rows may not include a timestamp,
# so all time-based analysis must be interpreted as based on the dated subset only.
df["comment_datetime_parsed"] = pd.NaT
for date_col in ["comment_datetime", "comment_published", "comment_date"]:
    if date_col in df.columns:
        parsed = pd.to_datetime(df[date_col], errors="coerce", utc=True)
        df["comment_datetime_parsed"] = df["comment_datetime_parsed"].fillna(parsed)

# Derived date fields
if df["comment_datetime_parsed"].notna().any():
    df["comment_date_parsed"] = df["comment_datetime_parsed"].dt.date
    df["comment_month_parsed"] = df["comment_datetime_parsed"].dt.to_period("M").astype(str)
else:
    df["comment_date_parsed"] = pd.NaT
    df["comment_month_parsed"] = np.nan

# Word/character length metrics
if "raw_word_count" not in df.columns:
    df["raw_word_count"] = df["text_readable"].apply(lambda x: len(str(x).split()))
if "raw_char_count" not in df.columns:
    df["raw_char_count"] = df["text_readable"].apply(lambda x: len(str(x)))

df["clean_word_count"] = df["cleaned_text"].apply(lambda x: len(str(x).split()))
df["clean_char_count"] = df["cleaned_text"].apply(lambda x: len(str(x)))

# Short labels for charts
def short_label(text, width=70):
    text = str(text).replace("\n", " ").strip()
    return shorten(text, width=width, placeholder="...")

df["post_title_short"] = df["post_title"].apply(lambda x: short_label(x, 70))

dated_df = df[df["comment_datetime_parsed"].notna()].copy()

print("Total cleaned comments:", len(df))
print("Comments with parsed dates:", len(dated_df))
print("Comments without parsed dates:", len(df) - len(dated_df))
print("Date coverage (%):", round(len(dated_df) / len(df) * 100, 2))

print("\nSample cleaned rows for report screenshots:")
display(df[["post_title", "text_readable", "cleaned_text", "raw_word_count", "clean_word_count"]].head(10))


In [ ]:

# ============================================================
# Cell 4: Final summary statistics table for the Data section
# ============================================================

summary_metrics = []

summary_metrics.append(["Final number of cleaned comments", len(df)])
summary_metrics.append(["Number of unique Reddit posts", df["post_link"].nunique() if "post_link" in df.columns else df["post_title"].nunique()])
summary_metrics.append(["Number of unique post titles", df["post_title"].nunique()])
summary_metrics.append(["Number of unique anonymised authors", df["comment_author_id"].nunique()])
summary_metrics.append(["Average raw comment length (characters)", round(df["raw_char_count"].mean(), 2)])
summary_metrics.append(["Median raw comment length (characters)", round(df["raw_char_count"].median(), 2)])
summary_metrics.append(["Average raw word count", round(df["raw_word_count"].mean(), 2)])
summary_metrics.append(["Median raw word count", round(df["raw_word_count"].median(), 2)])
summary_metrics.append(["Shortest raw comment length (characters)", int(df["raw_char_count"].min())])
summary_metrics.append(["Longest raw comment length (characters)", int(df["raw_char_count"].max())])
summary_metrics.append(["Average cleaned word count", round(df["clean_word_count"].mean(), 2)])
summary_metrics.append(["Median cleaned word count", round(df["clean_word_count"].median(), 2)])
summary_metrics.append(["Comments with parsed dates", len(dated_df)])
summary_metrics.append(["Comments without parsed dates", len(df) - len(dated_df)])
summary_metrics.append(["Date coverage (%)", round(len(dated_df) / len(df) * 100, 2)])

if len(dated_df) > 0:
    summary_metrics.append(["Earliest parsed comment date", str(dated_df["comment_datetime_parsed"].min())])
    summary_metrics.append(["Latest parsed comment date", str(dated_df["comment_datetime_parsed"].max())])

summary_table = pd.DataFrame(summary_metrics, columns=["Metric", "Value"])
summary_table.to_csv("outputs/final_summary_statistics.csv", index=False)

display(summary_table)
print("Saved: outputs/final_summary_statistics.csv")


In [ ]:

# ============================================================
# Cell 5: Helper functions and custom stopwords
# ============================================================

CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS).union({
    # general conversational words
    "just", "like", "think", "really", "know", "people", "time", "thing", "things",
    "make", "makes", "got", "going", "don", "didn", "doesn", "isn", "wasn", "im",
    "ive", "youre", "theyre", "cant", "would", "could", "should", "also", "much",
    "even", "one", "two", "way", "say", "said", "see", "seen", "get", "go", "went",
    "good", "bad", "right", "actually", "maybe", "probably", "yeah", "lot", "love",
    # platform/case-study terms that dominate but add little analytical detail
    "taylor", "swift", "taylors", "reddit", "swiftie", "swifties",
    # filler/short words often found after cleaning
    "lol", "omg", "haha", "yes", "no", "amp"
})


def tokens_from_series(series):
    """Tokenise cleaned text and remove short/custom stopwords."""
    tokens = []
    for text in series.fillna("").astype(str):
        for token in text.split():
            token = re.sub(r"[^a-zA-Z]", "", token).lower().strip()
            if len(token) > 2 and token not in CUSTOM_STOPWORDS:
                tokens.append(token)
    return tokens


def save_current_figure(filename):
    """Save the current matplotlib figure and display it."""
    path = f"figures/{filename}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")



## Exploratory Analysis

The next five outputs are the figures/tables for the Exploratory Analysis. Each chart is saved in the `figures/` folder.


In [ ]:

# ============================================================
# Exploratory Visual 1: Comments over time
# ============================================================

if len(dated_df) > 0:
    daily_comments = dated_df.groupby("comment_date_parsed").size().reset_index(name="comment_count")
    daily_comments.to_csv("outputs/eda_1_comments_over_time.csv", index=False)

    plt.figure(figsize=(10, 5))
    plt.plot(daily_comments["comment_date_parsed"], daily_comments["comment_count"], marker="o")
    plt.title("EDA 1: Taylor Swift Reddit comments over time (dated subset)")
    plt.xlabel("Comment date")
    plt.ylabel("Number of comments")
    plt.xticks(rotation=45, ha="right")
    save_current_figure("eda_1_comments_over_time")

    display(daily_comments.tail(15))
else:
    print("No parsed comment dates are available, so this time-series visual cannot be created.")


In [ ]:

# ============================================================
# Exploratory Visual 2: Top Reddit posts by number of comments
# ============================================================

top_posts = (
    df.groupby("post_title")
      .size()
      .sort_values(ascending=False)
      .head(10)
      .reset_index(name="comment_count")
)
top_posts["post_title_short"] = top_posts["post_title"].apply(lambda x: short_label(x, 80))
top_posts.to_csv("outputs/eda_2_top_posts_by_comments.csv", index=False)

plot_data = top_posts.sort_values("comment_count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_data["post_title_short"], plot_data["comment_count"])
plt.title("EDA 2: Top Reddit posts by number of collected comments")
plt.xlabel("Number of comments")
plt.ylabel("Post title")
save_current_figure("eda_2_top_posts_by_comments")

display(top_posts)


In [ ]:

# ============================================================
# Exploratory Visual 3: Top 20 most frequent cleaned words
# ============================================================

all_tokens = tokens_from_series(df["cleaned_text"])
word_counts = Counter(all_tokens)

top_words = pd.DataFrame(word_counts.most_common(20), columns=["word", "frequency"])
top_words.to_csv("outputs/eda_3_top_20_words.csv", index=False)

plot_data = top_words.sort_values("frequency", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_data["word"], plot_data["frequency"])
plt.title("EDA 3: Top 20 frequent words after cleaning")
plt.xlabel("Frequency")
plt.ylabel("Word")
save_current_figure("eda_3_top_20_words")

display(top_words)


In [ ]:

# ============================================================
# Exploratory Visual 4: Comment length distribution
# ============================================================

length_summary = df["raw_word_count"].describe().reset_index()
length_summary.columns = ["Statistic", "Raw word count"]
length_summary.to_csv("outputs/eda_4_comment_length_summary.csv", index=False)

# Clip very long comments at 200 words so the chart is readable.
# The full min/max values remain available in the summary table.
word_count_clipped = df["raw_word_count"].clip(upper=200)

plt.figure(figsize=(9, 5))
plt.hist(word_count_clipped, bins=40)
plt.title("EDA 4: Distribution of Reddit comment length")
plt.xlabel("Raw word count per comment (values above 200 clipped for readability)")
plt.ylabel("Number of comments")
save_current_figure("eda_4_comment_length_distribution")

display(length_summary)


In [ ]:

# ============================================================
# Exploratory Visual 5: Top active anonymised authors
# ============================================================

top_authors = (
    df["comment_author_id"]
      .value_counts()
      .head(15)
      .reset_index()
)
top_authors.columns = ["anonymised_author_id", "comment_count"]
top_authors.to_csv("outputs/eda_5_top_active_authors.csv", index=False)

plot_data = top_authors.sort_values("comment_count", ascending=True)

plt.figure(figsize=(9, 6))
plt.barh(plot_data["anonymised_author_id"], plot_data["comment_count"])
plt.title("EDA 5: Top 15 active anonymised Reddit authors")
plt.xlabel("Number of comments")
plt.ylabel("Anonymised author ID")
save_current_figure("eda_5_top_active_authors")

display(top_authors)


## Text Mining

The next section produces sentiment analysis and topic modelling outputs. VADER is used because it is widely used for short social-media-style text and provides a simple positive/neutral/negative classification.


In [ ]:

# ============================================================
# Cell 6: VADER sentiment scoring
# ============================================================

sia = SentimentIntensityAnalyzer()

# Use readable original text where available because VADER handles punctuation, emojis and intensifiers better than heavily cleaned text.
df["sentiment_source_text"] = np.where(
    df["text_readable"].str.len() > 0,
    df["text_readable"],
    df["comment_body"]
)

df["vader_compound"] = df["sentiment_source_text"].apply(lambda x: sia.polarity_scores(str(x))["compound"])

def vader_label(score):
    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"


df["sentiment_label"] = df["vader_compound"].apply(vader_label)

sentiment_summary = (
    df["sentiment_label"]
      .value_counts()
      .rename_axis("sentiment")
      .reset_index(name="comment_count")
)
sentiment_summary["percentage"] = round(sentiment_summary["comment_count"] / len(df) * 100, 2)
sentiment_summary.to_csv("outputs/text_1_sentiment_distribution.csv", index=False)

display(sentiment_summary)

# Save enriched dataset for later report writing and checking examples
sentiment_output_path = "outputs/taylor_swift_reddit_with_sentiment.csv"
df.to_csv(sentiment_output_path, index=False)
print(f"Saved enriched dataset: {sentiment_output_path}")


In [ ]:

# ============================================================
# Text Mining Visual 1: Sentiment distribution
# ============================================================

sentiment_order = ["Positive", "Neutral", "Negative"]
plot_data = sentiment_summary.set_index("sentiment").reindex(sentiment_order).dropna().reset_index()

plt.figure(figsize=(7, 5))
plt.bar(plot_data["sentiment"], plot_data["comment_count"])
plt.title("Text Mining 1: VADER sentiment distribution")
plt.xlabel("Sentiment category")
plt.ylabel("Number of comments")
save_current_figure("text_1_sentiment_distribution")

display(plot_data)


In [ ]:

# ============================================================
# Text Mining Visual 2: Sentiment over time
# ============================================================

sentiment_dated = df[df["comment_datetime_parsed"].notna()].copy()

if len(sentiment_dated) > 0:
    monthly_sentiment = (
        sentiment_dated.groupby("comment_month_parsed")
        .agg(
            average_compound_sentiment=("vader_compound", "mean"),
            comment_count=("vader_compound", "size")
        )
        .reset_index()
    )
    monthly_sentiment.to_csv("outputs/text_2_sentiment_over_time.csv", index=False)

    plt.figure(figsize=(9, 5))
    plt.plot(monthly_sentiment["comment_month_parsed"], monthly_sentiment["average_compound_sentiment"], marker="o")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.title("Text Mining 2: Average VADER sentiment over time (dated subset)")
    plt.xlabel("Month")
    plt.ylabel("Average compound sentiment")
    plt.xticks(rotation=45, ha="right")
    save_current_figure("text_2_sentiment_over_time")

    display(monthly_sentiment)
else:
    print("No parsed dates are available, so sentiment-over-time cannot be created.")


In [ ]:

# ============================================================
# Text Mining Visual/Table 3: Top positive and negative terms
# ============================================================

positive_tokens = tokens_from_series(df.loc[df["sentiment_label"] == "Positive", "cleaned_text"])
negative_tokens = tokens_from_series(df.loc[df["sentiment_label"] == "Negative", "cleaned_text"])

positive_terms = pd.DataFrame(Counter(positive_tokens).most_common(10), columns=["positive_term", "positive_frequency"])
negative_terms = pd.DataFrame(Counter(negative_tokens).most_common(10), columns=["negative_term", "negative_frequency"])

positive_negative_terms = pd.concat([positive_terms, negative_terms], axis=1)
positive_negative_terms.to_csv("outputs/text_3_top_positive_negative_terms.csv", index=False)

display(positive_negative_terms)

# Save the table as an image for easy insertion into the report.
fig, ax = plt.subplots(figsize=(11, 4))
ax.axis("off")
table = ax.table(
    cellText=positive_negative_terms.fillna("").values,
    colLabels=positive_negative_terms.columns,
    loc="center",
    cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.4)
plt.title("Text Mining 3: Top terms in positive and negative comments")
save_current_figure("text_3_top_positive_negative_terms")


In [ ]:

# ============================================================
# Cell 7: LDA topic modelling - compare 3, 5 and 10 topics
# ============================================================

# LDA works on document-term counts. Bigrams are included to capture phrases such as "era tour" and "music video".
vectorizer = CountVectorizer(
    max_df=0.90,
    min_df=5,
    max_features=5000,
    stop_words=list(CUSTOM_STOPWORDS),
    ngram_range=(1, 2)
)

document_term_matrix = vectorizer.fit_transform(df["cleaned_text"].fillna(""))
feature_names = vectorizer.get_feature_names_out()

lda_results = []
models = {}

for k in [3, 5, 10]:
    lda_model = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method="batch",
        max_iter=20
    )
    lda_model.fit(document_term_matrix)
    perplexity = lda_model.perplexity(document_term_matrix)
    models[k] = lda_model
    lda_results.append([k, round(perplexity, 2)])

lda_comparison = pd.DataFrame(lda_results, columns=["number_of_topics", "perplexity_lower_is_better"])
lda_comparison.to_csv("outputs/text_4_lda_model_comparison.csv", index=False)

display(lda_comparison)
print("Note: Perplexity is useful, but the final topic number should also be chosen for interpretability in the report.")


In [ ]:

# ============================================================
# Cell 8: Fit final 5-topic LDA model and label topics
# ============================================================

FINAL_K = 5
lda_final = models[FINAL_K]

def get_top_topic_terms(model, feature_names, n_terms=12):
    rows = []
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[:-n_terms - 1:-1]
        top_terms = [feature_names[i] for i in top_indices]
        rows.append({
            "topic_number": topic_idx + 1,
            "top_terms": ", ".join(top_terms)
        })
    return pd.DataFrame(rows)

topic_terms = get_top_topic_terms(lda_final, feature_names, n_terms=12)

def auto_topic_label(terms):
    terms_lower = terms.lower()
    if any(term in terms_lower for term in ["tour", "concert", "ticket", "era tour"]):
        return "Tour, concerts and live-event fandom"
    if any(term in terms_lower for term in ["red", "folklore", "evermore", "lover", "midnight", "reputation", "speak"]):
        return "Albums, eras and discography"
    if any(term in terms_lower for term in ["vault", "track", "version", "rank"]):
        return "Track rankings and re-recordings"
    if any(term in terms_lower for term in ["music", "video", "pop", "country", "fan"]):
        return "Music style, videos and fandom discussion"
    if any(term in terms_lower for term in ["ttpd", "poet", "tortured", "lyric", "bridge"]):
        return "TTPD, lyrics and song interpretation"
    return "General Taylor Swift discussion"

# These labels are manually interpretable and may be edited after inspecting top_terms.
topic_terms["suggested_topic_label"] = topic_terms["top_terms"].apply(auto_topic_label)

topic_terms.to_csv("outputs/text_5_lda_topic_terms_and_labels.csv", index=False)
display(topic_terms)

# Assign each comment to its dominant topic
topic_probabilities = lda_final.transform(document_term_matrix)
df["dominant_topic_number"] = topic_probabilities.argmax(axis=1) + 1

topic_label_map = dict(zip(topic_terms["topic_number"], topic_terms["suggested_topic_label"]))
df["dominant_topic_label"] = df["dominant_topic_number"].map(topic_label_map)

df.to_csv("outputs/taylor_swift_reddit_with_sentiment_and_topics.csv", index=False)
print("Saved enriched dataset with sentiment and topics: outputs/taylor_swift_reddit_with_sentiment_and_topics.csv")


In [ ]:

# ============================================================
# Text Mining Visual 4: Dominant topic distribution
# ============================================================

topic_distribution = (
    df["dominant_topic_label"]
    .value_counts()
    .reset_index()
)
topic_distribution.columns = ["topic_label", "comment_count"]
topic_distribution["percentage"] = round(topic_distribution["comment_count"] / len(df) * 100, 2)
topic_distribution.to_csv("outputs/text_6_dominant_topic_distribution.csv", index=False)

plot_data = topic_distribution.sort_values("comment_count", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_data["topic_label"], plot_data["comment_count"])
plt.title("Text Mining 4: Dominant discussion themes from LDA topic modelling")
plt.xlabel("Number of comments")
plt.ylabel("Dominant topic label")
save_current_figure("text_4_dominant_topic_distribution")

display(topic_distribution)


In [ ]:

# ============================================================
# Text Mining Visual 5: Average sentiment by topic
# ============================================================

sentiment_by_topic = (
    df.groupby("dominant_topic_label")
    .agg(
        comment_count=("vader_compound", "size"),
        average_compound_sentiment=("vader_compound", "mean"),
        positive_comments=("sentiment_label", lambda x: (x == "Positive").sum()),
        neutral_comments=("sentiment_label", lambda x: (x == "Neutral").sum()),
        negative_comments=("sentiment_label", lambda x: (x == "Negative").sum()),
    )
    .reset_index()
)

sentiment_by_topic["average_compound_sentiment"] = sentiment_by_topic["average_compound_sentiment"].round(3)
sentiment_by_topic.to_csv("outputs/text_7_sentiment_by_topic.csv", index=False)

plot_data = sentiment_by_topic.sort_values("average_compound_sentiment", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_data["dominant_topic_label"], plot_data["average_compound_sentiment"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Text Mining 5: Average VADER sentiment by LDA topic")
plt.xlabel("Average compound sentiment")
plt.ylabel("Dominant topic label")
save_current_figure("text_5_sentiment_by_topic")

display(sentiment_by_topic)


In [ ]:

# ============================================================
# Cell 9: Export top example comments for report interpretation
# ============================================================

# These examples help write the report, but do not include too many full Reddit comments in the final submission.
# Use short paraphrases or short quoted examples only where necessary.

top_positive_examples = (
    df.sort_values("vader_compound", ascending=False)
    [["post_title", "text_readable", "vader_compound", "sentiment_label", "dominant_topic_label"]]
    .head(10)
)

top_negative_examples = (
    df.sort_values("vader_compound", ascending=True)
    [["post_title", "text_readable", "vader_compound", "sentiment_label", "dominant_topic_label"]]
    .head(10)
)

top_positive_examples.to_csv("outputs/top_positive_comment_examples.csv", index=False)
top_negative_examples.to_csv("outputs/top_negative_comment_examples.csv", index=False)

print("Top positive examples:")
display(top_positive_examples)

print("Top negative examples:")
display(top_negative_examples)

print("Saved example tables in outputs/.")
